# 04 · Ciencia de datos alineada con la estadística

Cada hallazgo responde o matiza una pregunta del análisis estadístico:
1. **Calidad de datos:** qué hubo que corregir y cómo eso define la población.
2. **¿Cuándo existe la huella digital?** Por mes y por antigüedad: las banderas no se calcularon en julio, agosto
   y octubre, y casi no existen en líneas nuevas. Esto condiciona las proporciones de la sección 5.10.4.
3. **Segmentación en dos etapas:** con / sin huella digital, y K-means dentro de quienes tienen huella. Cada
   segmento se describe con las variables del documento (cargo fijo, antigüedad, banderas, restricción).
4. **Modelo de afinidad financiera digital** (banca, billetera o pagos en línea) y su efecto en la probabilidad
   hipergeométrica de una selección para campaña.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from scipy import stats
import config
import catalogo as cat

base = config.leer("base_trabajo")
el = config.leer("elegibles").reset_index(drop=True)
prep = config.leer_json("r00_preparacion")
constantes = {"BAN_" + c for c in prep["constantes"]}
FINANCIERAS = ["BAN_BANCA", "BAN_BILLETERA", "BAN_PAGOS_ONLINE"]
interes = [cat.COL(b) for f in cat.FAMILIAS_INTERES for b in cat.FAMILIAS[f]
           if cat.COL(b) in el and cat.COL(b) not in constantes and b not in cat.EXCLUIR]
el["huella"] = el[interes].sum(axis=1) > 0
el["financiera"] = el[FINANCIERAS].any(axis=1)
el["tramo"] = pd.cut(el.ANTIGUEDAD_MESES, [-1, 3, 12, 36, 120, 10_000],
                     labels=["0–3 meses", "4–12 meses", "13–36 meses", "37–120 meses", "Más de 120 meses"]).astype("string")
print(f"Banderas de interés útiles: {len(interes)} · registros con alguna: {el.huella.mean():.1%}")

Banderas de interés útiles: 42 · registros con alguna: 10.1%


## 1. Calidad de datos (base de trabajo, 9.000 registros)

In [2]:
def pct(mask):
    return float(mask.mean())


calidad = pd.DataFrame([
    ("EDAD = 0 en cédulas (edad faltante codificada como cero)", pct((base.EDAD == 0) & (base.TIPO_IDENTIFICACION == "Cedula")), "Excluidos de la población"),
    ("Registros de empresas (NIT)", pct(base.TIPO_IDENTIFICACION == "Nit"), "Excluidos: la población son personas naturales"),
    ("Estrato sin dato ('Se')", pct(base.ESTRATO == "Se"), "Variable descartada"),
    ("Tecnología del equipo sin dato", pct(base.TECNOLOGIA_TERMINAL_TOP == "SIN TECNOLOGIA"), "Variables de terminal descartadas"),
    ("Región comercial vacía", pct(base.REGION_COMERCIAL_TOP.isna()), "Se usa departamento"),
    ("Género sin dato ('ND')", pct(base.GENERO == "ND"), "Solo descriptivo"),
    ("Banderas constantes (35 de 88 columnas)", 35 / 88, "Excluidas del modelado"),
    ("Registros con atributos idénticos a otro (ID distinto)", prep["atributos_identicos"] / len(base), "Se conservan: son líneas distintas"),
    ("Sin ninguna bandera de interés (elegibles)", pct(~el.huella), "Ver hallazgo 2"),
], columns=["problema", "proporcion", "decision"])
calidad.sort_values("proporcion", ascending=False)

,problema,proporcion,decision
4,Región comercial vacía,1.000000,Se usa departamento
8,Sin ninguna bandera de interés (elegibles),0.899426,Ver hallazgo 2
2,Estrato sin dato ('Se'),0.869556,Variable descartada
3,Tecnología del equipo sin dato,0.839667,Variables de terminal descartadas
6,Banderas constantes (35 de 88 columnas),0.397727,Excluidas del modelado
5,Género sin dato ('ND'),0.369444,Solo descriptivo
1,Registros de empresas (NIT),0.208778,Excluidos: la población son personas naturales
0,EDAD = 0 en cédulas (edad faltante codificada ...,0.207444,Excluidos de la población
7,Registros con atributos idénticos a otro (ID d...,0.098889,Se conservan: son líneas distintas


## 2. ¿Cuándo existe la huella digital?

In [3]:
por_mes = (el.groupby("FECHA_ACTUALIZACION")
             .agg(registros=("huella", "size"), con_huella=("huella", "mean"), banca=("BAN_BANCA", "mean"),
                  billetera=("BAN_BILLETERA", "mean"), pagos=("BAN_PAGOS_ONLINE", "mean"),
                  antig_mediana=("ANTIGUEDAD_MESES", "median"), restriccion=("BAN_SUPRESION_I", "mean"))
             .reset_index())
por_mes["mes"] = por_mes.FECHA_ACTUALIZACION.map(config.MESES)
MEDIDOS = por_mes.loc[por_mes.con_huella > 0.02, "FECHA_ACTUALIZACION"].tolist()
print("Meses con huella medida:", [config.MESES[m] for m in MEDIDOS])
med = el.FECHA_ACTUALIZACION.isin(MEDIDOS)
por_tramo = (el[med].groupby("tramo").agg(registros=("huella", "size"), con_huella=("huella", "mean"),
                                         banca=("BAN_BANCA", "mean")).reindex(
    ["0–3 meses", "4–12 meses", "13–36 meses", "37–120 meses", "Más de 120 meses"]).reset_index())
prop_medidos = {b: float(el.loc[med, b].mean()) for b in FINANCIERAS + ["BAN_SUPRESION_I"]}
prop_medidos["n"] = int(med.sum())
print("Proporciones solo en meses medidos:", {k: round(v, 4) for k, v in prop_medidos.items()})
por_mes.round(3)

Meses con huella medida: ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio']
Proporciones solo en meses medidos: {'BAN_BANCA': 0.1075, 'BAN_BILLETERA': 0.0354, 'BAN_PAGOS_ONLINE': 0.0623, 'BAN_SUPRESION_I': 0.8748, 'n': 3162}


,FECHA_ACTUALIZACION,registros,con_huella,banca,billetera,pagos,antig_mediana,restriccion,mes
0,2026-01,501,0.178,0.112,0.034,0.066,3.0,0.872,Enero
1,2026-02,376,0.218,0.154,0.061,0.098,5.0,0.883,Febrero
2,2026-03,455,0.242,0.154,0.051,0.086,4.0,0.822,Marzo
3,2026-04,573,0.185,0.122,0.040,0.063,1.0,0.846,Abril
4,2026-05,726,0.056,0.025,0.010,0.021,1.0,0.934,Mayo
5,2026-06,531,0.177,0.128,0.036,0.070,1.0,0.866,Junio
6,2026-07,733,0.000,0.000,0.000,0.000,1.0,0.812,Julio
7,2026-08,947,0.004,0.001,0.000,0.001,1.0,0.895,Agosto
8,2026-10,388,0.000,0.000,0.000,0.000,2.0,0.964,Octubre


In [4]:
por_tramo.round(3)

,tramo,registros,con_huella,banca
0,0–3 meses,2118,0.050,0.036
1,4–12 meses,623,0.151,0.114
2,13–36 meses,139,0.827,0.561
3,37–120 meses,138,0.775,0.507
4,Más de 120 meses,141,0.702,0.305


## 3. Segmentación en dos etapas
**Etapa 1:** con / sin huella digital. **Etapa 2:** K-means sobre los índices de familia de quienes tienen huella
(fracción de banderas activas por familia). k se elige por silueta entre 3 y 6.

In [5]:
fam_cols = {}
for fam in cat.FAMILIAS_INTERES:
    cols = [c for c in interes if c[4:] in cat.FAMILIAS[fam]]
    if cols:
        fam_cols[fam] = cols
        el[f"idx_{fam}"] = el[cols].mean(axis=1)
idx = [f"idx_{f}" for f in fam_cols]
con = el[el.huella].copy()
Z = StandardScaler().fit_transform(con[idx])
sel = []
for k in range(3, 7):
    km = KMeans(n_clusters=k, n_init=20, random_state=config.SEED).fit(Z)
    sel.append({"k": k, "silueta": float(silhouette_score(Z, km.labels_)), "inercia": float(km.inertia_)})
sel = pd.DataFrame(sel)
K = int(sel.sort_values("silueta", ascending=False).iloc[0].k)
km = KMeans(n_clusters=K, n_init=50, random_state=config.SEED).fit(Z)
con["seg"] = km.labels_
pca = PCA(n_components=3, random_state=config.SEED).fit(Z)
coords = pca.transform(Z)
print(f"k elegido = {K} · varianza explicada PCA: {pca.explained_variance_ratio_.round(3)}")
sel.round(3)

k elegido = 3 · varianza explicada PCA: [0.489 0.135 0.115]


,k,silueta,inercia
0,3,0.317,1968.254
1,4,0.291,1632.727
2,5,0.297,1439.664
3,6,0.261,1298.402


In [6]:
# Nombres con reglas explícitas sobre el perfil de cada grupo (documentadas para poder discutirlas):
# - banca digital ≥ 60 % y sin sobrerrepresentación tecnológica → "Digitales financieros"
# - índice de tecnología ≥ 3 veces el promedio del grupo con huella → "Hogar conectado (IoT)"
# - el resto, con huella pero por debajo del promedio en casi todas las familias → "Huella básica: redes y familia"
media_con = con[idx].mean()
nombres = {}
for s_, g in con.groupby("seg"):
    lift_tec = g["idx_tecnologia"].mean() / media_con["idx_tecnologia"] if "idx_tecnologia" in g else 0
    if lift_tec >= 3:
        nombres[s_] = "Hogar conectado (IoT)"
    elif g.BAN_BANCA.mean() >= 0.6:
        nombres[s_] = "Digitales financieros"
    else:
        nombres[s_] = "Huella básica: redes y familia"
con["segmento"] = con.seg.map(nombres)
el["segmento"] = "Sin huella digital"
el.loc[con.index, "segmento"] = con.segmento


def perfil(g):
    cfm = g.VAL_CFM.astype(float)
    return pd.Series({
        "registros": len(g), "pct": len(g) / len(el),
        "cfm_media": cfm.mean(), "cfm_mediana": cfm.median(), "cfm_sd": cfm.std(ddof=1),
        "cfm_cv": cfm.std(ddof=1) / cfm.mean() * 100,
        "antig_mediana": g.ANTIGUEDAD_MESES.median(), "antig_media": g.ANTIGUEDAD_MESES.mean(),
        "banca": g.BAN_BANCA.mean(), "billetera": g.BAN_BILLETERA.mean(), "pagos": g.BAN_PAGOS_ONLINE.mean(),
        "restriccion": g.BAN_SUPRESION_I.mean(), "edad_media": g.EDAD.mean(),
    })


perfiles = el.groupby("segmento").apply(perfil, include_groups=False).sort_values("registros", ascending=False)
afin = []
for s_, g in con.groupby("segmento"):
    lift = (g[interes].mean() / con[interes].mean().replace(0, np.nan)).dropna()
    lift = lift[con[interes].mean() >= 0.03].sort_values(ascending=False).head(4)
    afin.append({"segmento": s_, "afinidades": [[c[4:].lower(), round(float(v) * 100)] for c, v in lift.items()],
                 "familias": {f: round(float(g[f"idx_{f}"].mean() / media_con[f"idx_{f}"] * 100)) for f in fam_cols}})
perfiles.round(3)

,registros,pct,cfm_media,cfm_mediana,cfm_sd,cfm_cv,antig_mediana,antig_media,banca,billetera,pagos,restriccion,edad_media
segmento,,,,,,,,,,,,,
Sin huella digital,4704.0,0.899,39593.406,40185.0,5298.834,13.383,1.0,4.156,0.000,0.000,0.000,0.928,38.419
Digitales financieros,280.0,0.054,39324.718,38339.0,10665.338,27.121,28.0,47.525,0.857,0.329,0.582,0.393,44.218
Huella básica: redes y familia,218.0,0.042,32244.417,36288.5,13350.792,41.405,29.5,63.968,0.358,0.046,0.096,0.450,49.174
Hogar conectado (IoT),28.0,0.005,40294.464,38339.0,8962.698,22.243,14.5,34.786,0.821,0.357,0.500,0.357,40.393


## 4. Modelo de afinidad financiera digital
Variable respuesta: banca, billetera **o** pagos en línea activos. Se entrena solo con los meses en que la huella
se midió (sin julio, agosto ni octubre), para no enseñarle al modelo "ceros" que en realidad son datos no calculados.

Se usa un **árbol de decisión de 3 niveles**: cada hoja es una regla del tipo "si … entonces P(afinidad) = …",
es decir, una **probabilidad condicional** como las del Tema 3. Se compara con un gradient boosting para mostrar
un riesgo frecuente: un modelo complejo puede "acertar" casi siempre solo porque detecta si la línea tiene huella.

In [7]:
from sklearn.tree import DecisionTreeClassifier

dm = el[med].reset_index(drop=True)
no_fin = [c for c in interes if c[4:] not in cat.FAMILIAS["finanzas"]]
X = pd.concat([dm[["VAL_CFM", "ANTIGUEDAD_MESES", "EDAD"]].astype(float).fillna(-1),
               dm[no_fin + ["BAN_SUPRESION_I"]].astype(int),
               (dm.SEGMENTO_CLIENTE == "Pyme").astype(int).rename("SEGMENTO_PYME")], axis=1)
y = dm.financiera.astype(int)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=config.SEED)
arbol = DecisionTreeClassifier(max_depth=3, min_samples_leaf=30, random_state=config.SEED)
oof = cross_val_predict(arbol, X, y, cv=cv, method="predict_proba")[:, 1]
aucs = [roc_auc_score(y.iloc[te], oof[te]) for _, te in cv.split(X, y)]
auc, ap, tasa = roc_auc_score(y, oof), average_precision_score(y, oof), y.mean()
gb = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=15, random_state=config.SEED)
auc_gb = roc_auc_score(y, cross_val_predict(gb, X, y, cv=cv, method="predict_proba")[:, 1])
print(f"n = {len(y):,} · positivos {y.sum()} ({tasa:.1%}) · árbol AUC-ROC {auc:.3f} "
      f"(pliegues {np.mean(aucs):.3f} ± {np.std(aucs):.3f}) · AUC-PR {ap:.3f} · gradient boosting AUC {auc_gb:.3f}")

n = 3,162 · positivos 366 (11.6%) · árbol AUC-ROC 0.988 (pliegues 0.986 ± 0.005) · AUC-PR 0.882 · gradient boosting AUC 0.990


### Evaluación con datos de entrenamiento y prueba (guía del proyecto, punto 5)
Se separa el 70 % de los registros para entrenar y el 30 % para probar, manteniendo la misma proporción de
positivos en ambos (estratificado). Se reporta exactitud y F1 sobre la prueba, junto con lo que lograría un modelo
"ingenuo" que siempre dice "no": con solo 11,6 % de positivos, la exactitud sola engaña.

In [8]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.30, stratify=y, random_state=config.SEED)
arbol_p = DecisionTreeClassifier(max_depth=3, min_samples_leaf=30, random_state=config.SEED).fit(X_ent, y_ent)
pred = arbol_p.predict(X_pru)
mc = confusion_matrix(y_pru, pred, labels=[0, 1])
prueba = {"n_entrenamiento": int(len(y_ent)), "n_prueba": int(len(y_pru)), "positivos_prueba": int(y_pru.sum()),
          "exactitud": float(accuracy_score(y_pru, pred)), "precision": float(precision_score(y_pru, pred, zero_division=0)),
          "sensibilidad": float(recall_score(y_pru, pred)), "f1": float(f1_score(y_pru, pred)),
          "auc_prueba": float(roc_auc_score(y_pru, arbol_p.predict_proba(X_pru)[:, 1])),
          "exactitud_ingenuo": float(1 - y_pru.mean()),
          "matriz": {"vn": int(mc[0, 0]), "fp": int(mc[0, 1]), "fn": int(mc[1, 0]), "vp": int(mc[1, 1])}}
pd.Series(prueba).to_frame("valor")

,valor
n_entrenamiento,2213
n_prueba,949
positivos_prueba,110
exactitud,0.970495
precision,0.820312
sensibilidad,0.954545
f1,0.882353
auc_prueba,0.990595
exactitud_ingenuo,0.884089
matriz,"{'vn': 816, 'fp': 23, 'fn': 5, 'vp': 105}"


In [9]:
arbol.fit(X, y)
t = arbol.tree_
nombres_x = list(X.columns)


LEGIBLE = {"COMIDAYBEBIDA": "comida y bebida", "COMPRASYMODA": "compras y moda", "REDES_SOCIALES": "redes sociales",
           "APPS_TRANSPORTE": "apps de transporte", "TEC_IOT": "dispositivos IoT", "CLASE_MEDIA": "clase media",
           "CLASE_ALTA": "clase alta", "TEC_ELECTRODOMESTICOS": "electrodomésticos inteligentes"}


def legible(var, umbral, izquierda):
    if var.startswith("BAN_"):
        nombre = LEGIBLE.get(var[4:], var[4:].lower().replace("_", " "))
        if var == "BAN_SUPRESION_I":
            return "sin restricción de contacto" if izquierda else "con restricción de contacto"
        return f"sin {nombre}" if izquierda else f"con {nombre}"
    unidad = {"VAL_CFM": "cargo fijo", "ANTIGUEDAD_MESES": "antigüedad (meses)", "EDAD": "edad"}.get(var, var)
    return f"{unidad} ≤ {umbral:,.0f}" if izquierda else f"{unidad} > {umbral:,.0f}"


reglas = []


def recorrer(nodo, condiciones):
    if t.children_left[nodo] == -1:
        valores = t.value[nodo][0]
        n_hoja = int(t.n_node_samples[nodo])
        p_hoja = float(valores[1] / valores.sum())
        reglas.append({"regla": " y ".join(condiciones) or "todos", "registros": n_hoja, "p_afinidad": p_hoja,
                       "positivos": round(p_hoja * n_hoja), "lift": p_hoja / tasa})
        return
    var, umbral = nombres_x[t.feature[nodo]], t.threshold[nodo]
    recorrer(t.children_left[nodo], condiciones + [legible(var, umbral, True)])
    recorrer(t.children_right[nodo], condiciones + [legible(var, umbral, False)])


recorrer(0, [])
reglas = pd.DataFrame(reglas).sort_values("p_afinidad", ascending=False)
importancia = (pd.Series(arbol.feature_importances_, index=X.columns).sort_values(ascending=False)
                 .head(8).rename_axis("variable").rename("importancia").reset_index())
importancia = importancia[importancia.importancia > 0]
dm["score"] = oof
dm["decil"] = pd.qcut(dm.score.rank(method="first", ascending=False), 10, labels=range(1, 11)).astype(int)
deciles = dm.groupby("decil").agg(registros=("financiera", "size"), positivos=("financiera", "sum")).reset_index()
deciles["tasa"] = deciles.positivos / deciles.registros
deciles["lift"] = deciles.tasa / tasa
deciles["captura"] = deciles.positivos.cumsum() / deciles.positivos.sum()
reglas.round(3)

,regla,registros,p_afinidad,positivos,lift
5,con familia y con hogar y con comida y bebida,120,0.983,118,8.495
4,con familia y con hogar y sin comida y bebida,195,0.795,155,6.867
2,con familia y sin hogar y antigüedad (meses) ≤...,123,0.634,78,5.479
3,con familia y sin hogar y antigüedad (meses) >...,30,0.200,6,1.728
1,sin familia y con redes sociales,53,0.170,9,1.467
0,sin familia y sin redes sociales,2641,0.000,0,0.000


### Conexión con la hipergeométrica (5.10.5)
¿Qué probabilidad hay de que, al escoger 20 registros para una campaña de servicios financieros, al menos 5 tengan
afinidad financiera? Se compara escoger al azar entre todos los registros de los meses medidos frente a escoger al
azar dentro de la regla del árbol con mayor probabilidad.

In [10]:
m_sel = 20
mejor = reglas.iloc[0]
hoja_mejor = arbol.apply(X) == arbol.apply(X)[np.argmax(arbol.predict_proba(X)[:, 1])]
hip = {}
for nombre, u in [("Al azar entre todos", dm), (f"Al azar en la mejor regla", dm[hoja_mejor])]:
    Nn, Kk = len(u), int(u.financiera.sum())
    rv = stats.hypergeom(Nn, Kk, min(m_sel, Nn))
    hip[nombre] = {"N": Nn, "K": Kk, "m": m_sel, "esperado": float(rv.mean()), "P_al_menos_5": float(rv.sf(4)),
                   "P_al_menos_10": float(rv.sf(9)), "dist": [float(rv.pmf(x)) for x in range(m_sel + 1)]}
hip_regla = mejor.regla
pd.DataFrame({k_: {c: v for c, v in d.items() if c != "dist"} for k_, d in hip.items()}).T

,N,K,m,esperado,P_al_menos_5,P_al_menos_10
Al azar entre todos,3162.0,366.0,20.0,2.314991,0.072398,0.000024
Al azar en la mejor regla,120.0,118.0,20.0,19.666667,1.000000,1.000000


## 5. Hallazgos alineados con las preguntas estadísticas

In [11]:
s_med = perfiles.drop(index="Sin huella digital", errors="ignore")
alineacion = [
    {"pregunta": "5.3.1 · ¿Cómo se distribuye la antigüedad y qué grupos orientan la fidelización?",
     "hallazgo": f"Las líneas de 0–3 meses casi no tienen huella digital ({por_tramo.con_huella.iloc[0]:.0%} en meses medidos) frente a {por_tramo.con_huella.iloc[-1]:.0%} en las de más de 120 meses: la permanencia trae información para personalizar."},
    {"pregunta": "5.3.3–5.3.5 · ¿Qué proporción tiene banca, billetera y pagos en línea?",
     "hallazgo": f"Las proporciones de toda la muestra están subestimadas: las banderas no se calcularon en julio, agosto y octubre. En los meses medidos, banca digital sube de {el.BAN_BANCA.mean():.1%} a {prop_medidos['BAN_BANCA']:.1%}."},
    {"pregunta": "5.3.6 · ¿Cómo condiciona la restricción de contacto una selección?",
     "hallazgo": "La restricción y la falta de huella van juntas: entre contactables casi la mitad tiene huella; entre restringidos, 5 %. Una campaña digital debe partir de los contactables."},
    {"pregunta": "5.7.3 · ¿Qué grupos tienen mayor variación relativa del cargo fijo?",
     "hallazgo": f"El segmento con mayor CV del cargo fijo es {perfiles.cfm_cv.idxmax()} ({perfiles.cfm_cv.max():.1f} %); el de menor, {perfiles.cfm_cv.idxmin()} ({perfiles.cfm_cv.min():.1f} %)."},
    {"pregunta": "5.10.5 · ¿Qué probabilidad hay de seleccionar registros con una preferencia digital?",
     "hallazgo": f"Escogiendo 20 registros al azar, P(≥5 con afinidad financiera) = {hip['Al azar entre todos']['P_al_menos_5']:.1%}; escogiéndolos dentro de la regla «{hip_regla}», {hip['Al azar en la mejor regla']['P_al_menos_5']:.1%}."},
]
for a_ in alineacion:
    print("•", a_["pregunta"], "\n ", a_["hallazgo"])

• 5.3.1 · ¿Cómo se distribuye la antigüedad y qué grupos orientan la fidelización? 
  Las líneas de 0–3 meses casi no tienen huella digital (5% en meses medidos) frente a 70% en las de más de 120 meses: la permanencia trae información para personalizar.
• 5.3.3–5.3.5 · ¿Qué proporción tiene banca, billetera y pagos en línea? 
  Las proporciones de toda la muestra están subestimadas: las banderas no se calcularon en julio, agosto y octubre. En los meses medidos, banca digital sube de 6.5% a 10.8%.
• 5.3.6 · ¿Cómo condiciona la restricción de contacto una selección? 
  La restricción y la falta de huella van juntas: entre contactables casi la mitad tiene huella; entre restringidos, 5 %. Una campaña digital debe partir de los contactables.
• 5.7.3 · ¿Qué grupos tienen mayor variación relativa del cargo fijo? 
  El segmento con mayor CV del cargo fijo es Huella básica: redes y familia (41.4 %); el de menor, Sin huella digital (13.4 %).
• 5.10.5 · ¿Qué probabilidad hay de seleccionar regist

In [12]:
muestra3d = con.assign(pc1=coords[:, 0], pc2=coords[:, 1], pc3=coords[:, 2])[
    ["pc1", "pc2", "pc3", "segmento", "VAL_CFM", "ANTIGUEDAD_MESES", "BAN_SUPRESION_I"]]
config.guardar_json({
    "calidad": calidad.to_dict("records"), "por_mes": por_mes.to_dict("records"), "meses_medidos": MEDIDOS,
    "por_tramo": por_tramo.to_dict("records"), "prop_medidos": prop_medidos, "con_huella": float(el.huella.mean()),
    "huella_por_restriccion": {"contactables": float(el[el.BAN_SUPRESION_I == False].huella.mean()),   # noqa: E712
                               "restringidos": float(el[el.BAN_SUPRESION_I == True].huella.mean())},  # noqa: E712
    "n_interes": len(interes), "familias": {f: cat.NOMBRE_FAMILIA[f] for f in fam_cols},
    "seleccion_k": sel.to_dict("records"), "K": K, "varianza_pca": pca.explained_variance_ratio_.tolist(),
    "perfiles": perfiles.reset_index().to_dict("records"), "afinidad": afin,
    "puntos": {c: (muestra3d[c].round(3).tolist() if c.startswith("pc") else muestra3d[c].tolist()) for c in muestra3d},
    "modelo": {"n": len(y), "positivos": int(y.sum()), "tasa": float(tasa), "auc": auc, "auc_pliegues": aucs, "ap": ap,
               "auc_gb": auc_gb, "deciles": deciles.to_dict("records"), "importancia": importancia.to_dict("records"),
               "reglas": reglas.to_dict("records"), "variables": X.shape[1], "prueba": prueba},
    "hip_regla": hip_regla,
    "hipergeometrica": hip, "alineacion": alineacion,
}, "r04_ciencia_datos")